In [65]:
import keras
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [66]:
(train_input, train_target), _ = keras.datasets.fashion_mnist.load_data()
train_scaled = train_input / 255.0

In [67]:
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2
)

In [68]:
train_scaled.shape, val_scaled.shape

((48000, 28, 28), (12000, 28, 28))

In [69]:
def model_fn(a_layer=None):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(28,28)))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(100, activation='relu'))
    if a_layer:
        model.add(a_layer)
    model.add(keras.layers.Dense(10, activation='softmax'))

    return model

#### 드롭아웃

In [70]:
model = model_fn(a_layer=keras.layers.Dropout(0.3))
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
# checkpoint = keras.callbacks.ModelCheckpoint('best-model.keras',
#                                 save_best_only=True)

In [72]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",  # val_loss 모니터링 하다가
    patience=2,  # 2 에폭보다가 개선 없으면
    restore_best_weights=True,  # 가장 좋았던 때로 복원
)

In [73]:
model.compile(
    optimizer="adam",
    loss=keras.losses.sparse_categorical_crossentropy,
    metrics=[keras.metrics.sparse_categorical_accuracy],
)

history = model.fit(
    train_scaled,
    train_target,
    epochs=20,
    verbose=1,
    validation_data=(val_scaled, val_target),
    callbacks=[ early_stopping],  # callback 함수 + early stopping 추가
)

Epoch 1/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.5884 - sparse_categorical_accuracy: 0.7915 - val_loss: 0.4225 - val_sparse_categorical_accuracy: 0.8465
Epoch 2/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.4371 - sparse_categorical_accuracy: 0.8419 - val_loss: 0.3940 - val_sparse_categorical_accuracy: 0.8542
Epoch 3/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.4030 - sparse_categorical_accuracy: 0.8544 - val_loss: 0.3630 - val_sparse_categorical_accuracy: 0.8649
Epoch 4/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.3810 - sparse_categorical_accuracy: 0.8614 - val_loss: 0.3543 - val_sparse_categorical_accuracy: 0.8720
Epoch 5/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.3634 - sparse_categorical_accuracy: 0.8683 - val_loss: 0.3489 - val_sparse_categorical_accuracy: 0.8707
Epoch 6/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.3526 - sparse_categorical_accuracy: 0.8710 - val_loss: 0.3480 - val_sparse_categorical_accuracy:

In [74]:
print("stopped_epoch =", early_stopping.stopped_epoch)

stopped_epoch = 11


In [75]:
# model.save('model.keras')

In [76]:
# model.save_weights('model.weights.h5')